# Myanmar Programming Tutor — QLoRA Fine-Tuning (Kaggle)

**Base model:** `google/gemma-3-4b-it`  
**Method:** QLoRA 4-bit supervised fine-tuning  
**Data:** v0.2 gold `data/processed/chat/train.jsonl`, `val.jsonl` (textbook rewrite; not v0.1 extraction)

## Kaggle Setup
1. Upload this repo as a Kaggle Dataset, or clone from GitHub
2. Settings → **GPU** ON, **Internet** ON
3. Add Hugging Face token as Kaggle secret if needed for Gemma access
4. Run all cells sequentially

In [ ]:
!pip install -q -r ../requirements-kaggle.txt

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer, SFTConfig

# Adjust if repo mounted elsewhere on Kaggle
ROOT = Path("/kaggle/input/myanmar-programming-tutor") if Path("/kaggle/input/myanmar-programming-tutor").exists() else Path("..")
TRAIN_PATH = ROOT / "data/processed/chat/train.jsonl"
VAL_PATH = ROOT / "data/processed/chat/val.jsonl"
OUTPUT_DIR = Path("/kaggle/working/lora_adapter")
MODEL_NAME = "google/gemma-3-4b-it"

print(f"ROOT={ROOT}")
print(f"Train exists: {TRAIN_PATH.exists()}")

In [ ]:
def load_chat_jsonl(path: Path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def format_example(row, tokenizer):
    return tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False
    )


train_rows = load_chat_jsonl(TRAIN_PATH)
val_rows = load_chat_jsonl(VAL_PATH)
print(f"Train: {len(train_rows)}, Val: {len(val_rows)}")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
train_texts = [format_example(r, tokenizer) for r in train_rows]
val_texts = [format_example(r, tokenizer) for r in val_rows]

train_ds = Dataset.from_dict({"text": train_texts})
val_ds = Dataset.from_dict({"text": val_texts})

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    max_length=2048,
    bf16=True,
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(train_result)

In [ ]:
# Plot training loss
log_history = trainer.state.log_history
losses = [x["loss"] for x in log_history if "loss" in x]
steps = list(range(len(losses)))

plt.figure(figsize=(8, 4))
plt.plot(steps, losses)
plt.title("Training Loss")
plt.xlabel("Logged step index")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.savefig("/kaggle/working/training_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /kaggle/working/training_loss.png")

In [ ]:
# Sanity check: 3 validation prompts
model.eval()
for row in val_rows[:3]:
    prompt = tokenizer.apply_chat_template(
        row["messages"][:1], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.2, do_sample=True)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print("=" * 60)
    print(row["messages"][0]["content"][:200])
    print("-" * 60)
    print(text[-600:])